## **Articulo**

In [16]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
articulo = spark.read.table("lh_retailnova_bronze_dev.articulos")
display(articulo)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf1c74f8-0dba-4864-9a05-31f0bcf0cbb0)

In [17]:
%%sql
DROP TABLE IF EXISTS dim_categoria;
CREATE TABLE dim_categoria AS
SELECT
    ROW_NUMBER() OVER (ORDER BY id_categ_n2) AS id_categ_n2,
    id_categ_n2 AS categoria
FROM (
    SELECT DISTINCT id_categ_n2
    FROM lh_retailnova_bronze_dev.articulos
) ;

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 24, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
%%sql
select *
from dim_categoria

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 25, Finished, Available, Finished, False)

<Spark SQL result set with 19 rows and 2 fields>

In [19]:
%%sql
DROP TABLE IF EXISTS dim_macro_categoria;
CREATE TABLE dim_macro_categoria AS
SELECT
    ROW_NUMBER() OVER (ORDER BY id_categ_n1) AS id_categ_n1,
    id_categ_n1 AS macro_categoria
FROM (
    SELECT DISTINCT id_categ_n1
    FROM lh_retailnova_bronze_dev.articulos
) ;

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 27, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [20]:
%%sql
select *
from dim_macro_categoria

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 28, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 2 fields>

In [21]:
%%sql
DROP TABLE IF EXISTS dim_sub_categoria;
CREATE TABLE dim_sub_categoria AS
SELECT
    ROW_NUMBER() OVER (ORDER BY id_categ_n3) AS id_categ_n3,
    id_categ_n3 AS sub_categoria
FROM (
    SELECT DISTINCT id_categ_n3
    FROM lh_retailnova_bronze_dev.articulos
) ;

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 30, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
%%sql
select *
from dim_sub_categoria

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 31, Finished, Available, Finished, False)

<Spark SQL result set with 59 rows and 2 fields>

In [23]:
%%sql
DROP TABLE IF EXISTS articulo;
CREATE TABLE articulo AS
SELECT
    -- Llaves de Dimensiones
    
    dmc.id_categ_n1,
    dc.id_categ_n2,
    dsc.id_categ_n3,
    
    -- Campos y Métricas de la Transacción
   
    art.articulo_id,
    art.id_proveedor,
    art.precio_lista,
    art.peso_kg,
    art.unid_medida,
    art.activo,
    art.fec_alta
   
   
FROM lh_retailnova_bronze_dev.articulos art


-- 2. Joins con las Dimensiones de Categoría de acuerdo con los niveles del artículo
JOIN dim_macro_categoria dmc
    ON art.id_categ_n1 = dmc.macro_categoria

JOIN dim_categoria dc
    ON art.id_categ_n2 = dc.categoria

JOIN dim_sub_categoria dsc
    ON art.id_categ_n3 = dsc.sub_categoria



StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 33, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
%%sql
SELECT *
from articulo

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 34, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 10 fields>

In [25]:
# Eliminar duplicados basándose estrictamente en el ID de la transacción
articulo = articulo.dropDuplicates(["articulo_id"])


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 35, Finished, Available, Finished, False)

In [26]:
(articulo.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.articulos") # <-- Destino corregido
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 36, Finished, Available, Finished, False)

## **DEVOLUCION**

In [27]:
from pyspark.sql.types import *
devolucion= spark.read.table("lh_retailnova_bronze_dev.devolucion")
display(devolucion)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 183bc5ed-664c-45bb-83de-3e8b58a7afd2)

In [28]:
(devolucion.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.hechos_devolucion")
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 38, Finished, Available, Finished, False)

## **STOCK**

In [29]:
# Eliminar duplicados basándose estrictamente en el ID de la transacción
articulo = articulo.dropDuplicates(["articulo_id"])


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 39, Finished, Available, Finished, False)

In [30]:
from pyspark.sql.types import *
stock = spark.read.table("lh_retailnova_bronze_dev.stock")
display(stock)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b428f5d4-b565-43ed-a187-9b076b5863b7)

In [31]:
# Eliminar duplicados basándose estrictamente en el ID de la transacción
articulo = articulo.dropDuplicates(["articulo_id"])


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 41, Finished, Available, Finished, False)

In [32]:
(stock.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.hechos_stock")
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 42, Finished, Available, Finished, False)

## **VENTA**

In [33]:
from pyspark.sql.types import *
venta = spark.read.table("lh_retailnova_bronze_dev.venta")
display(venta)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fac8be5f-cdf5-4140-8f6c-e2c2e1d35344)

In [35]:
# Eliminar duplicados basándose estrictamente en el ID de la transacción
venta = venta.dropDuplicates(["id_transaccion"])


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 45, Finished, Available, Finished, False)

In [37]:
(venta.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.hechos_transaccion")
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 47, Finished, Available, Finished, False)

## **MIEMBROS**

In [38]:
from pyspark.sql.types import *
miembros = spark.read.table("lh_retailnova_bronze_dev.miembros")
display(miembros)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2115ea3d-ebea-41a4-9c34-6b663d9bfdcf)

In [39]:
miembros = (miembros
    .dropDuplicates(["id_miembro"])
    .dropna() 
    .withColumn("fec_registro", F.col("fecha_registro").cast("date"))
    .withColumn("activo", F.col("activo").cast("boolean"))
    .withColumn("fec_ultima_compra", F.col("fecha_ultima_compra").cast("date"))
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 49, Finished, Available, Finished, False)

In [40]:
from pyspark.sql.functions import sha2, concat_ws
import pyspark.sql.functions as F


miembros = miembros.withColumn("id_miembro_hash", sha2(F.col("id_miembro").cast("string"), 256))

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 50, Finished, Available, Finished, False)

In [41]:
display(miembros)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 494fc317-ebb0-429a-a64e-b6f78a184915)

In [42]:
total_entrantes = miembros.count()


print("         REPORTE DE CALIDAD - CAPA SILVER        ")

print(f"Total Registros Evaluados : {total_entrantes}")

print("--------------------------------------------------")
print("Porcentaje de valores nulos por columna:")

for columna in miembros.columns:
    nulos_columna = miembros.filter(F.col(columna).isNull()).count()
    pct_nulos = round((nulos_columna / total_entrantes) * 100, 2) if total_entrantes > 0 else 0
    print(f" - {columna}: {pct_nulos}% nulos ({nulos_columna} registros)")


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 52, Finished, Available, Finished, False)

         REPORTE DE CALIDAD - CAPA SILVER        
Total Registros Evaluados : 47566
--------------------------------------------------
Porcentaje de valores nulos por columna:
 - id_miembro: 0.0% nulos (0 registros)
 - canal_preferido: 0.0% nulos (0 registros)
 - genero: 0.0% nulos (0 registros)
 - ciudad: 0.0% nulos (0 registros)
 - fecha_registro: 0.0% nulos (0 registros)
 - fecha_ultima_compra: 0.0% nulos (0 registros)
 - edad: 0.0% nulos (0 registros)
 - activo: 0.0% nulos (0 registros)
 - fec_registro: 0.0% nulos (0 registros)
 - fec_ultima_compra: 0.0% nulos (0 registros)
 - id_miembro_hash: 0.0% nulos (0 registros)


In [43]:
(miembros.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.dim_miembros")
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 53, Finished, Available, Finished, False)

In [44]:
display(miembros)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fbf1a961-1a1e-47af-84c7-51fb274c676c)

## **PROVEEDORES**

In [45]:
from pyspark.sql.types import *
proveedor = spark.read.table("lh_retailnova_bronze_dev.proveedores")
display(proveedor)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f9e6ff7-4d38-44d4-8b67-59f542605d65)

In [47]:
# Eliminar duplicados basándose estrictamente en el ID de la transacción
proveedor = proveedor.dropDuplicates(["id_proveedor"])


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 57, Finished, Available, Finished, False)

In [48]:
proveedor = ( proveedor
    .dropDuplicates(["id_proveedor"])
    .withColumn("tiempo_repo_dias", F.col("tiempo_repo_dias").cast("int"))
    .withColumn("calificacion_calidad", F.col("calificacion_calidad").cast("double"))
    .withColumn("activo", F.col("activo").cast("boolean"))
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 58, Finished, Available, Finished, False)

In [49]:
(proveedor.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.dim_proveedor")
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 59, Finished, Available, Finished, False)

## **TIENDA**

In [50]:
from pyspark.sql.types import *
tienda = spark.read.table("lh_retailnova_bronze_dev.tienda")
display(tienda)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d1b27273-05dc-4832-ac33-d8ac72324a95)

In [51]:
%%sql
DROP TABLE IF EXISTS dim_pais;
CREATE TABLE dim_pais AS
SELECT
    ROW_NUMBER() OVER (ORDER BY pais ) AS id_pais,
pais 
FROM (
    SELECT DISTINCT pais
    FROM lh_retailnova_bronze_dev.tienda
) ;

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 62, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [52]:
%%sql
SELECT *
from dim_pais

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 63, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 2 fields>

In [53]:
%%sql
DROP TABLE IF EXISTS dim_ciudad;
CREATE TABLE dim_ciudad AS
SELECT
    ROW_NUMBER() OVER (ORDER BY ciudad) AS id_ciudad,
    ciudad  
FROM (
    SELECT DISTINCT ciudad
    FROM lh_retailnova_bronze_dev.tienda
) ;

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 65, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [54]:

   
 %%sql
DROP TABLE IF EXISTS tienda;
CREATE TABLE tienda AS
SELECT
    -- Llaves de Dimensiones
    
    tie.id_tienda,
    tie.tipo_tienda,
    tie.nom_tienda,
    tie.fecha_apertura,
    tie.metros_cuadrados,
    tie.activo,
    dp.id_pais,
    dc.id_ciudad
   
FROM   lh_retailnova_bronze_dev.tienda tie


-- 2. Joins con las Dimensiones de Categoría de acuerdo con los niveles del artículo
JOIN dim_pais dp
    ON dp.pais = tie.pais

JOIN dim_ciudad dc
    ON tie.ciudad = dc.ciudad





StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 67, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [55]:
tienda= (tienda
    .dropDuplicates(["id_tienda"])
    .dropna() 
   
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 68, Finished, Available, Finished, False)

In [56]:
total_entrantes = tienda.count()


print("         REPORTE DE CALIDAD - CAPA SILVER        ")

print(f"Total Registros Evaluados : {total_entrantes}")

print("--------------------------------------------------")
print("Porcentaje de valores nulos por columna:")

for columna in tienda.columns:
    nulos_columna = tienda.filter(F.col(columna).isNull()).count()
    pct_nulos = round((nulos_columna / total_entrantes) * 100, 2) if total_entrantes > 0 else 0
    print(f" - {columna}: {pct_nulos}% nulos ({nulos_columna} registros)")


StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 69, Finished, Available, Finished, False)

         REPORTE DE CALIDAD - CAPA SILVER        
Total Registros Evaluados : 141
--------------------------------------------------
Porcentaje de valores nulos por columna:
 - id_tienda: 0.0% nulos (0 registros)
 - tipo_tienda: 0.0% nulos (0 registros)
 - nom_tienda: 0.0% nulos (0 registros)
 - pais: 0.0% nulos (0 registros)
 - ciudad: 0.0% nulos (0 registros)
 - fecha_apertura: 0.0% nulos (0 registros)
 - metros_cuadrados: 0.0% nulos (0 registros)
 - activo: 0.0% nulos (0 registros)


In [57]:
(tienda.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("lh_retailnova_silver_dev.dim_tienda")
)

StatementMeta(, 7fb442b5-e61a-417e-a5d6-f59207c92dde, 70, Finished, Available, Finished, False)